# LLM-from-Scratch (100M Parameters)

Train a 100M parameter GPT-style Transformer from scratch on Google Colab T4.

## 1. Verify GPU & Install Dependencies

In [ ]:
!pip install -q torch numpy tqdm requests matplotlib regex

import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"PyTorch: {torch.__version__}")

## 2. Mount Google Drive (Optional but Recommended)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!mkdir -p /content/drive/MyDrive/llm-from-scratch/checkpoints
!mkdir -p /content/drive/MyDrive/llm-from-scratch/logs

## 3. Clone Repository

In [ ]:
%cd /content
!git clone https://github.com/avneeshjadhav04/llm-from-scratch.git
%cd llm-from-scratch

## 4. Prepare Data (Train BPE Tokenizer)

In [ ]:
!python prepare_data.py

## 5. Train the 100M Model

> **Session limit:** We cap each Colab session to **5,000 steps** (~45–50 min) so we finish well within the 12-hour wall and can save checkpoints to Drive.
> Adjust `SESSION_STEPS` as needed.

In [ ]:
SESSION_STEPS = 5000
!python train.py --max_steps_per_session {SESSION_STEPS}

## 6. Save Checkpoints to Drive

In [ ]:
import shutil
import os

for f in os.listdir("checkpoints"):
    if f.endswith(".pt"):
        shutil.copy(f"checkpoints/{f}", "/content/drive/MyDrive/llm-from-scratch/checkpoints/")
        print(f"Saved: {f}")

for f in os.listdir("logs"):
    shutil.copy(f"logs/{f}", "/content/drive/MyDrive/llm-from-scratch/logs/")

## 7. Generate Text

> Uses the **latest saved checkpoint**. Update the path if you want a specific step.

In [ ]:
import glob
ckpts = sorted(glob.glob("checkpoints/*.pt"))
if ckpts:
    latest_ckpt = ckpts[-1]
    !python generate.py \
        --checkpoint {latest_ckpt} \
        --prompt "The future of artificial intelligence is" \
        --temperature 0.8 \
        --top_k 50 \
        --max_new_tokens 256 \
        --device cuda
else:
    print("No checkpoint found. Train first.")

## 8. Resume Training from Drive Checkpoint

In [ ]:
!cp /content/drive/MyDrive/llm-from-scratch/checkpoints/*.pt checkpoints/ 2>/dev/null || true
!cp /content/drive/MyDrive/llm-from-scratch/logs/* logs/ 2>/dev/null || true

SESSION_STEPS = 5000
!python train.py --max_steps_per_session {SESSION_STEPS}